# FDS Pokémon Battle Prediction

This notebook clones our repository and rebuilds our three selected submissions
from the competition files. Each rebuilt CSV is compared with the file we submitted, which is stored in
`results/submitted/`. On our machine the rebuilt files are byte-identical; on Kaggle's CPUs and Python 3.13 a
few tree splits can differ, so the notebook reports how many of the 5,000 predictions match. Both the rebuilt and
the submitted files are written to the output.

| Submission | Public | Private | Model |
|---|---:|---:|---|
| `submission_matched_core_summary.csv` | 0.70466 | 0.69371 | Five-view tree model, original features |
| `submission_team_similarity.csv` | 0.70200 | 0.69514 | Five-view tree model, corrected features, plus similar-team records |
| `submission_tactical_team_spikes.csv` | 0.70000 | 0.70200 | Tactical view only, corrected features, plus similar-team records and Spikes |

Two full training runs are needed (original and corrected features). The third
submission reuses the second run. With 2 workers each run takes about 2.5 hours, so the notebook needs about 5 hours.

Attach the competition data and enable Internet access. Use a fresh session.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

repo_url = "https://github.com/facelessM/FDS-Pokemon-Challenge-2026-2027.git"
branch = "main"
repo = Path("/kaggle/working/FDS-Pokemon-Challenge-2026-2027")
env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    env["FDS_GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # A public repository can be cloned without a secret.

if not repo.exists():
    with tempfile.TemporaryDirectory() as folder:
        askpass = Path(folder) / "askpass.py"
        askpass.write_text(
            f"#!{sys.executable}\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] '
            'else os.environ.get("FDS_GITHUB_TOKEN", ""))\n'
        )
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
        result = subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(repo)],
                                env=env, capture_output=True)
        if result.returncode:
            raise RuntimeError("Clone failed. Check Internet access and GITHUB_TOKEN.")
env.pop("FDS_GITHUB_TOKEN", None)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())

In [ ]:
# Install before importing any of the model libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)

In [ ]:
import hashlib
import json
import shutil

import pandas as pd

matches = sorted(Path("/kaggle/input").rglob("train.jsonl"))
data_candidates = [p.parent for p in matches if (p.parent / "test.jsonl").exists()
                   and list(p.parent.glob("*type_chart*.csv"))]
if len(data_candidates) != 1:
    raise RuntimeError("Expected one attached competition dataset; set data explicitly if needed.")
data = data_candidates[0]
outputs = Path("/kaggle/working/outputs")
selected = json.loads((repo / "results/selected_submissions.json").read_text())


def run(*args):
    subprocess.run([sys.executable, "run.py", *args, "--data", str(data)], check=True, cwd=repo)


def deliver(name, path):
    """Compare a rebuilt CSV with the submitted one and copy both to /kaggle/working.

    On the machine we used, the rebuilt file is byte-identical to the submitted
    one. Other CPUs or Python versions can change a few tree splits, so here we
    report how many of the 5,000 predictions agree instead of stopping.
    """
    rebuilt = pd.read_csv(path)
    submitted_path = repo / "results/submitted" / path.name
    submitted = pd.read_csv(submitted_path)
    assert (rebuilt["battle_id"] == submitted["battle_id"]).all()
    same = int((rebuilt["player_won"] == submitted["player_won"]).sum())
    identical = hashlib.sha256(path.read_bytes()).hexdigest() == selected[name]["sha256"]
    shutil.copyfile(path, Path("/kaggle/working") / path.name)
    shutil.copyfile(submitted_path, Path("/kaggle/working") / f"submitted_{path.name}")
    status = "identical to the submitted file" if identical else (
        f"{same} of {len(submitted)} predictions ({same / len(submitted):.2%}) match the submitted file")
    print(f"{path.name}: {status} "
          f"(public {selected[name]['public_score']}, private {selected[name]['private_score']})")

## Submission 1: five-view model (public 0.70466, private 0.69371)

Five feature views, each a LightGBM and XGBoost pair (raw observations use
LightGBM only), averaged with equal weights:

1. Reconstructed battle state at turn 30, recovery resources, likely P2 knockouts and team/move records.
2. The same state with species-specific HP and status and a learned five-turn HP forecast.
3. The same state with possible move sets and how many answers each side still has (tactical view).
4. The same state with learned pressure for attacks whose recorded base power is zero.
5. Raw summaries of the observed turns.

We fit one ensemble per fold of three grouped 5-fold partitions. Each test battle
gets one prediction per partition, and the three are averaged before the 0.5
threshold. Mean partition CV 0.7060, averaged out-of-fold accuracy 0.7103.
This run uses the features as they were when we made the submission.

In [ ]:
run("predict", "--output", str(outputs / "original"), "--jobs", "2")
deliver("matched_core_summary", outputs / "original" / "submission_matched_core_summary.csv")

## Submission 2: corrected features plus similar-team records (public 0.70200, private 0.69514)

After the first submission we found four errors in feature construction and
fixed them with `--feature-fixes` (details in `src/options.py`): move accuracy was
read as a percentage, P2 knockout scores were not calibrated, knockout risk only
reached common species, and validation rows saw estimators fitted on more data
than training rows.

A residual scan then showed that the trees underused one signal: how often teams
similar to P1's team (sharing at least four species) won, and the same for teams
containing every Pokémon P2 has revealed. A logistic regression combines the
five-view probability with these two rates. It is trained only on out-of-fold
probabilities. Mean partition CV 0.7080, averaged out-of-fold accuracy 0.7118.

In [ ]:
run("predict", "--feature-fixes", "--output", str(outputs / "fixed"), "--jobs", "2")
run("stage", "--output", str(outputs / "fixed"), "--view", "blend", "--signals", "team", "--name", "team_similarity")
deliver("team_similarity", outputs / "fixed" / "submission_team_similarity.csv")

## Submission 3: one view plus two signals (public 0.70000, private 0.70200)

The simplest model we found that matches the five-view ensemble: only the
tactical view (one LightGBM and XGBoost pair), then the same logistic second
stage with the similar-team rates and one more signal, Spikes still down on each
side after turn 30 (Rapid Spin removals included). Mean partition CV 0.7077,
averaged out-of-fold accuracy 0.7085, and the best log loss of our models (0.5497).
It reuses the corrected run from submission 2.

In [ ]:
run("stage", "--output", str(outputs / "fixed"), "--view", "tactical", "--signals", "team,spikes", "--name", "tactical_team_spikes")
deliver("tactical_team_spikes", outputs / "fixed" / "submission_tactical_team_spikes.csv")